# Assignment 3: PySpark Filter, GroupBy, Aggregations & Sorting
**Subject:** Big Data Systems

**Topics Covered:**
1. Filter Operation
2. Logical Operators (AND `&`, OR `|`)
3. Comparison Operators (`==`, `!=`)
4. GroupBy
5. Overall Aggregated Values
6. `format_number` and `alias`
7. Sort using `orderBy`

## Step 1: Install PySpark

In [1]:
!pip install pyspark

## Step 2: Import Libraries

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    sum   as spark_sum,
    avg,
    count,
    max   as spark_max,
    min   as spark_min,
    format_number,
    round as spark_round
)
from pyspark.sql.types import (
    StructType, StructField, IntegerType, StringType, FloatType
)

print("Libraries imported successfully.")

Libraries imported successfully.


## Step 3: Start Spark Session

In [3]:
spark = SparkSession.builder \
    .appName("Assignment3_Filter_GroupBy_Aggregation") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark Session started.")

Spark Session started.


## Step 4: Load Dataset
We use a **Retail Sales** dataset with 25 records (created inline).
Columns: `order_id`, `product`, `category`, `region`, `sales_rep`, `units_sold`, `unit_price`, `total_revenue`, `discount_pct`, `profit`

In [4]:
schema = StructType([
    StructField("order_id",      IntegerType(), False),
    StructField("product",       StringType(),  False),
    StructField("category",      StringType(),  False),
    StructField("region",        StringType(),  False),
    StructField("sales_rep",     StringType(),  False),
    StructField("units_sold",    IntegerType(), False),
    StructField("unit_price",    FloatType(),   False),
    StructField("total_revenue", FloatType(),   False),
    StructField("discount_pct",  FloatType(),   True),
    StructField("profit",        FloatType(),   False),
])

data = [
    (1001, "Laptop Pro",        "Electronics", "North", "Rahul",  5,  55000.0, 275000.0,  5.0, 38500.0),
    (1002, "Wireless Mouse",    "Accessories", "South", "Priya",  20,   850.0,  17000.0, 10.0,  5100.0),
    (1003, "Office Chair",      "Furniture",   "West",  "Amit",   8,  12500.0, 100000.0,  0.0, 22000.0),
    (1004, "Smartphone X",      "Electronics", "East",  "Neha",   12, 30000.0, 360000.0,  8.0, 57600.0),
    (1005, "Desk Lamp",         "Accessories", "North", "Rahul",  35,  1200.0,  42000.0,  5.0,  9450.0),
    (1006, "Gaming Keyboard",   "Accessories", "South", "Priya",  15,  3500.0,  52500.0,  0.0, 15750.0),
    (1007, "Standing Desk",     "Furniture",   "North", "Vikram",  4, 25000.0, 100000.0, 10.0, 18000.0),
    (1008, "Tablet Lite",       "Electronics", "West",  "Amit",    9, 22000.0, 198000.0,  5.0, 31680.0),
    (1009, "USB Hub",           "Accessories", "East",  "Neha",   50,   550.0,  27500.0,  0.0,  9625.0),
    (1010, "Monitor 4K",        "Electronics", "North", "Vikram",  7, 35000.0, 245000.0,  5.0, 36750.0),
    (1011, "Bookshelf",         "Furniture",   "South", "Priya",   6,  8500.0,  51000.0,  0.0, 12750.0),
    (1012, "Headphones Pro",    "Accessories", "West",  "Amit",   18,  4500.0,  81000.0,  8.0, 19440.0),
    (1013, "Smart Watch",       "Electronics", "East",  "Rahul",  10, 18000.0, 180000.0,  5.0, 27000.0),
    (1014, "Laptop Stand",      "Accessories", "North", "Vikram", 25,  2200.0,  55000.0,  0.0, 19250.0),
    (1015, "Ergonomic Chair",   "Furniture",   "South", "Neha",    3, 18000.0,  54000.0, 10.0, 10260.0),
    (1016, "Bluetooth Speaker", "Electronics", "West",  "Priya",  14,  6500.0,  91000.0,  0.0, 27300.0),
    (1017, "Filing Cabinet",    "Furniture",   "North", "Rahul",   5,  9500.0,  47500.0,  5.0, 10687.0),
    (1018, "Webcam HD",         "Accessories", "East",  "Amit",   30,  2800.0,  84000.0,  5.0, 25200.0),
    (1019, "Laptop Pro",        "Electronics", "South", "Vikram",  3, 55000.0, 165000.0, 10.0, 24750.0),
    (1020, "Wireless Charger",  "Accessories", "West",  "Neha",   40,  1500.0,  60000.0,  0.0, 21000.0),
    (1021, "Conference Table",  "Furniture",   "East",  "Rahul",   2, 45000.0,  90000.0,  5.0, 17100.0),
    (1022, "Smartwatch Lite",   "Electronics", "North", "Priya",  11, 12000.0, 132000.0,  8.0, 21120.0),
    (1023, "Pen Drive 128GB",   "Accessories", "South", "Amit",   60,   650.0,  39000.0,  0.0, 14625.0),
    (1024, "Sofa Set",          "Furniture",   "West",  "Vikram",  2, 65000.0, 130000.0,  5.0, 24700.0),
    (1025, "Smartphone Y",      "Electronics", "East",  "Neha",    8, 25000.0, 200000.0, 10.0, 30000.0),
]

df = spark.createDataFrame(data, schema=schema)
print("Dataset Loaded Successfully!")
print(f"Total Records: {df.count()}")
df.show(truncate=False)

Dataset Loaded Successfully!
Total Records: 25
+--------+-----------------+-----------+------+---------+----------+----------+-------------+------------+-------+
|order_id|product          |category   |region|sales_rep|units_sold|unit_price|total_revenue|discount_pct|profit |
+--------+-----------------+-----------+------+---------+----------+----------+-------------+------------+-------+
|1001    |Laptop Pro       |Electronics|North |Rahul    |5         |55000.0   |275000.0     |5.0         |38500.0|
|1002    |Wireless Mouse   |Accessories|South |Priya    |20        |850.0     |17000.0      |10.0        |5100.0 |
|1003    |Office Chair     |Furniture  |West  |Amit     |8         |12500.0   |100000.0     |0.0         |22000.0|
|1004    |Smartphone X     |Electronics|East  |Neha     |12        |30000.0   |360000.0     |8.0         |57600.0|
|1005    |Desk Lamp        |Accessories|North |Rahul    |35        |1200.0    |42000.0      |5.0         |9450.0 |
|1006    |Gaming Keyboard  |Acces

## 1. Filter Operation
Extract rows from the DataFrame based on a condition using `.filter()` or `.where()`.
- **Syntax:** `df.filter(condition)` or `df.where(condition)`

In [5]:
# Filter 1: Only Electronics category
print("-- Filter: Electronics category --")
df.filter(col("category") == "Electronics").show(truncate=False)

# Filter 2: Orders where total_revenue > 1,00,000
print("-- Filter: total_revenue > 1,00,000 --")
df.filter(col("total_revenue") > 100000).show(truncate=False)

-- Filter: Electronics category --
+--------+-----------------+-----------+------+---------+----------+----------+-------------+------------+-------+
|order_id|product          |category   |region|sales_rep|units_sold|unit_price|total_revenue|discount_pct|profit |
+--------+-----------------+-----------+------+---------+----------+----------+-------------+------------+-------+
|1001    |Laptop Pro       |Electronics|North |Rahul    |5         |55000.0   |275000.0     |5.0         |38500.0|
|1004    |Smartphone X     |Electronics|East  |Neha     |12        |30000.0   |360000.0     |8.0         |57600.0|
|1008    |Tablet Lite      |Electronics|West  |Amit     |9         |22000.0   |198000.0     |5.0         |31680.0|
|1010    |Monitor 4K       |Electronics|North |Vikram   |7         |35000.0   |245000.0     |5.0         |36750.0|
|1013    |Smart Watch      |Electronics|East  |Rahul    |10        |18000.0   |180000.0     |5.0         |27000.0|
|1016    |Bluetooth Speaker|Electronics|West 

## 2. Logical Operators
- **AND operator** `&` — both conditions must be true
- **OR operator** `|` — at least one condition must be true

> Always wrap each condition in **parentheses** `()` when using `&` or `|`.

In [6]:
# AND operator (&): Electronics AND revenue > 2,00,000
print("-- AND Operator: Electronics AND total_revenue > 2,00,000 --")
df.filter(
    (col("category") == "Electronics") & (col("total_revenue") > 200000)
).show(truncate=False)

# OR operator (|): Furniture OR revenue > 2,50,000
print("-- OR Operator: Furniture OR total_revenue > 2,50,000 --")
df.filter(
    (col("category") == "Furniture") | (col("total_revenue") > 250000)
).show(truncate=False)

-- AND Operator: Electronics AND total_revenue > 2,00,000 --
+--------+------------+-----------+------+---------+----------+----------+-------------+------------+-------+
|order_id|product     |category   |region|sales_rep|units_sold|unit_price|total_revenue|discount_pct|profit |
+--------+------------+-----------+------+---------+----------+----------+-------------+------------+-------+
|1001    |Laptop Pro  |Electronics|North |Rahul    |5         |55000.0   |275000.0     |5.0         |38500.0|
|1004    |Smartphone X|Electronics|East  |Neha     |12        |30000.0   |360000.0     |8.0         |57600.0|
|1010    |Monitor 4K  |Electronics|North |Vikram   |7         |35000.0   |245000.0     |5.0         |36750.0|
+--------+------------+-----------+------+---------+----------+----------+-------------+------------+-------+

-- OR Operator: Furniture OR total_revenue > 2,50,000 --
+--------+----------------+-----------+------+---------+----------+----------+-------------+------------+------

## 3. Comparison Operators
- `==` — Equal to
- `!=` — Not equal to
- `>`, `<`, `>=`, `<=` — Greater/Less than comparisons

In [7]:
# Equality (==): North region only
print("-- == Operator: Only North region --")
df.filter(col("region") == "North") \
  .select("order_id", "product", "region", "total_revenue") \
  .show()

# Not Equal (!=): Exclude Accessories
print("-- != Operator: All categories EXCEPT Accessories --")
df.filter(col("category") != "Accessories") \
  .select("order_id", "product", "category", "profit") \
  .show()

-- == Operator: Only North region --
+--------+---------------+------+-------------+
|order_id|        product|region|total_revenue|
+--------+---------------+------+-------------+
|    1001|     Laptop Pro| North|     275000.0|
|    1005|      Desk Lamp| North|      42000.0|
|    1007|  Standing Desk| North|     100000.0|
|    1010|     Monitor 4K| North|     245000.0|
|    1014|   Laptop Stand| North|      55000.0|
|    1017| Filing Cabinet| North|      47500.0|
|    1022|Smartwatch Lite| North|     132000.0|
+--------+---------------+------+-------------+

-- != Operator: All categories EXCEPT Accessories --
+--------+-----------------+-----------+-------+
|order_id|          product|   category| profit|
+--------+-----------------+-----------+-------+
|    1001|       Laptop Pro|Electronics|38500.0|
|    1003|     Office Chair|  Furniture|22000.0|
|    1004|     Smartphone X|Electronics|57600.0|
|    1007|    Standing Desk|  Furniture|18000.0|
|    1008|      Tablet Lite|Electronic

## 4. GroupBy
Group rows by one or more columns and apply aggregate functions.
- **Syntax:** `df.groupBy("column").agg(...)` or `df.groupBy("column").count()`

In [8]:
# GroupBy category — count orders per category
print("-- GroupBy category: Number of orders --")
df.groupBy("category").count() \
  .orderBy("count", ascending=False) \
  .show()

# GroupBy region — total revenue per region
print("-- GroupBy region: Total revenue --")
df.groupBy("region") \
  .agg(spark_sum("total_revenue").alias("Total_Revenue")) \
  .orderBy("Total_Revenue", ascending=False) \
  .show()

-- GroupBy category: Number of orders --
+-----------+-----+
|   category|count|
+-----------+-----+
|Electronics|    9|
|Accessories|    9|
|  Furniture|    7|
+-----------+-----+

-- GroupBy region: Total revenue --
+------+-------------+
|region|Total_Revenue|
+------+-------------+
|  East|     941500.0|
| North|     896500.0|
|  West|     660000.0|
| South|     378500.0|
+------+-------------+



## 5. Overall Aggregated Values
Apply aggregate functions (`sum`, `avg`, `max`, `min`, `count`) across the **entire** DataFrame without grouping.

In [9]:
overall = df.agg(
    spark_sum("total_revenue").alias("Total_Revenue_All"),
    avg("total_revenue").alias("Avg_Revenue"),
    spark_max("profit").alias("Max_Profit"),
    spark_min("profit").alias("Min_Profit"),
    count("*").alias("Total_Orders")
)

print("-- Overall Aggregated Statistics (entire dataset) --")
overall.show(truncate=False)

-- Overall Aggregated Statistics (entire dataset) --
+-----------------+-----------+----------+----------+------------+
|Total_Revenue_All|Avg_Revenue|Max_Profit|Min_Profit|Total_Orders|
+-----------------+-----------+----------+----------+------------+
|2876500.0        |115060.0   |57600.0   |5100.0    |25          |
+-----------------+-----------+----------+----------+------------+



## 6. format_number and alias
- **`format_number(col, decimal_places)`** — Formats a number with comma separators (e.g., 2,75,000.00)
- **`.alias("new_name")`** — Renames a column in the output

In [10]:
# format_number: display total revenue per category with comma formatting
formatted = df.groupBy("category") \
    .agg(spark_sum("total_revenue").alias("raw_total")) \
    .select(
        col("category"),
        format_number("raw_total", 2).alias("Total_Revenue_Formatted")
    ).orderBy("category")

print("-- Revenue per category (formatted with commas, 2 decimal places) --")
formatted.show(truncate=False)

# alias on a column expression
print("-- alias example: derived columns renamed --")
df.select(
    col("product"),
    (col("total_revenue") - col("profit")).alias("Total_Cost"),
    col("profit").alias("Net_Profit")
).show(5, truncate=False)

-- Revenue per category (formatted with commas, 2 decimal places) --
+-----------+-----------------------+
|category   |Total_Revenue_Formatted|
+-----------+-----------------------+
|Accessories|458,000.00             |
|Electronics|1,846,000.00           |
|Furniture  |572,500.00             |
+-----------+-----------------------+

-- alias example: derived columns renamed --
+--------------+----------+----------+
|product       |Total_Cost|Net_Profit|
+--------------+----------+----------+
|Laptop Pro    |236500.0  |38500.0   |
|Wireless Mouse|11900.0   |5100.0    |
|Office Chair  |78000.0   |22000.0   |
|Smartphone X  |302400.0  |57600.0   |
|Desk Lamp     |32550.0   |9450.0    |
+--------------+----------+----------+
only showing top 5 rows


## 7. Sort Data using orderBy
- **`orderBy(col.asc())`** — Ascending order (default)
- **`orderBy(col.desc())`** — Descending order
- Can sort by multiple columns at once

In [11]:
# Sort by total_revenue DESCENDING
print("-- Sort by total_revenue DESCENDING (highest first) --")
df.select("product", "category", "region", "total_revenue") \
  .orderBy(col("total_revenue").desc()) \
  .show(truncate=False)

# Sort by profit ASCENDING
print("-- Sort by profit ASCENDING (lowest first) --")
df.select("product", "units_sold", "profit") \
  .orderBy(col("profit").asc()) \
  .show(truncate=False)

# Multi-column sort: category ASC, then total_revenue DESC
print("-- Multi-column sort: category ASC, total_revenue DESC --")
df.select("product", "category", "total_revenue") \
  .orderBy(col("category").asc(), col("total_revenue").desc()) \
  .show(truncate=False)

-- Sort by total_revenue DESCENDING (highest first) --
+-----------------+-----------+------+-------------+
|product          |category   |region|total_revenue|
+-----------------+-----------+------+-------------+
|Smartphone X     |Electronics|East  |360000.0     |
|Laptop Pro       |Electronics|North |275000.0     |
|Monitor 4K       |Electronics|North |245000.0     |
|Smartphone Y     |Electronics|East  |200000.0     |
|Tablet Lite      |Electronics|West  |198000.0     |
|Smart Watch      |Electronics|East  |180000.0     |
|Laptop Pro       |Electronics|South |165000.0     |
|Smartwatch Lite  |Electronics|North |132000.0     |
|Sofa Set         |Furniture  |West  |130000.0     |
|Office Chair     |Furniture  |West  |100000.0     |
|Standing Desk    |Furniture  |North |100000.0     |
|Bluetooth Speaker|Electronics|West  |91000.0      |
|Conference Table |Furniture  |East  |90000.0      |
|Webcam HD        |Accessories|East  |84000.0      |
|Headphones Pro   |Accessories|West  |81000.

## Stop Spark Session

In [12]:
spark.stop()
print("Spark Session stopped. Assignment 3 complete.")

Spark Session stopped. Assignment 3 complete.
